# R4: Classify a New Month of Imagery

Runs the trained model over every temporal-RGB image in a month of the
archive and writes one CSV row per image.

You only need this notebook to process a month yourself. The December
2015 output (`data/StromNet_2015_12.csv`, 1,164,270 rows) is already
provided, so R5 runs without the image archive.


In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from.  All paths and constants live in
# config.py; edit that one file rather than anything below.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()


In [ ]:
import os, csv, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import tensorflow as tf

# ---- Edit these two lines for the month you want ----
MONTH_ROOT = config.RGB_ARCHIVE_DIR / "2015" / "12"          # <archive>/YYYY/MM
OUTPUT_CSV = config.OUTPUT_DIR / "StromNet_2015_12_rerun.csv"

BATCH_SIZE = 32
MODEL_PATH = config.MODEL_PATH   # or config.OUTPUT_DIR / "lava_lake_classifier_retrained.keras"

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Model loaded:", MODEL_PATH)


## 1. Build the (correctly ordered) file list

Sorting `dirnames` in place makes `os.walk` traverse day and hour folders
chronologically, so the CSV comes out time-ordered by construction rather
than by filesystem accident.

In [ ]:
def list_month_images(root):
    files = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        for name in sorted(filenames):
            if name.lower().endswith(".jpg"):
                files.append(os.path.join(dirpath, name))
    return files

file_list = list_month_images(MONTH_ROOT)
print(f"{len(file_list):,} images under {MONTH_ROOT}")
assert file_list, "No images found - check config.RGB_ARCHIVE_DIR and MONTH_ROOT."


## 2. Stream the images through the model

Images decode and resize lazily inside the `tf.data` pipeline; the model's
own `Rescaling` layer handles the 1/255 normalization (do not also divide
here). One `predict()` call processes the whole month; a full month
(~1.3 M images) yields only ~15 MB of probabilities.

In [ ]:
def load_img(path):
    img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, [config.IMG_HEIGHT, config.IMG_WIDTH])

ds = (tf.data.Dataset.from_tensor_slices(file_list)
      .map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
      .batch(BATCH_SIZE)
      .prefetch(tf.data.AUTOTUNE))

t0 = time.time()
probs = model.predict(ds, verbose=1)     # columns: [no, small, large]
print(f"Predicted {len(probs):,} images in {time.time() - t0:.0f} s")


## 3. Write the CSV

Model output order is `[no, small, large]`; the CSV stores
`timestamp, large, no, small` for compatibility with the provided
December 2015 file and with R5.

In [ ]:
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    for path, p in zip(file_list, probs):
        stem = os.path.splitext(os.path.basename(path))[0]
        writer.writerow([stem, p[2], p[0], p[1]])
print("Wrote", OUTPUT_CSV)
print("Point config.CSV_2015_12 (or R5's CSV_PATH override) at this file to analyze it.")
